# Study 1: interferometer placement by expected information gain

Reads the scan written by `run_study.py`: for every toroidal plane and chord direction, the best single chord and the best pair of parallel chords, scored by the expected Kullback-Leibler divergence about the `ne` parameters (bits, mean over five truths). Figures go to `plots/` as PDF.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from plasma_state_estimation_for_diagnostic_placement_optimization.bayesian_inference import (  # noqa: E501
    inference_types,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.diagnostics_forward_models import (  # noqa: E501
    geometry_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.mhd_equilibrium_forward_models import (  # noqa: E501
    dataset_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.results.diagnostics_placement_optimization.if_placement_study import (  # noqa: E501
    run_study,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    plots_diagnostics_forward_models,
    pubfig,
    sweep_utils,
)

pubfig.apply_style()

SCAN = run_study.OUTPUT
# Beside this notebook, whatever directory the kernel was started in.
PLOTS = SCAN.parents[1] / "plots"
PLOTS.mkdir(exist_ok=True)
PACKAGE = run_study.OUTPUT.parents[5]
PATHS = sweep_utils.SweepPaths(
    data_directory=PACKAGE.parents[1] / "data",
    dataset=dataset_utils.DEFAULT_DATASET,
    models_directory=PACKAGE / "mhd_equilibrium_forward_models" / "trained_models",
    model_name=inference_types.DEFAULT_SURROGATE_NAME,
)
table = pd.read_csv(SCAN)
best = table.loc[
    table.groupby(["toroidal_angle_deg", "n_chords"]).information_gain_bits.idxmax()
]
best.head()

## 1. Objective against toroidal angle

Best direction and offsets per plane, each curve divided by its own maximum, with the initial guess of section 2 in every plane (grey) divided by the same maximum.

In [ ]:
# The initial guess of section 2 in every plane, on the scale of the optimized curve.
initial_guesses = {
    n_chords: run_study.initial_guess(
        PATHS, tuple(float(angle) for angle in rows.toroidal_angle_deg), int(n_chords)
    )
    for n_chords, rows in best.groupby("n_chords")
}
figure, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for axis, (n_chords, rows) in zip(axes, best.groupby("n_chords")):
    gain = rows.information_gain_bits
    guess = initial_guesses[n_chords]
    axis.plot(
        guess.toroidal_angle_deg,
        guess.information_gain_bits / gain.max(),
        "o-",
        color="0.6",
        label="initial guess",
    )
    axis.plot(rows.toroidal_angle_deg, gain / gain.max(), "o-", label="optimized")
    axis.set_title(
        f"{n_chords} chord{'s' if n_chords > 1 else ''} (max {gain.max():.2f} bit)"
    )
    axis.set_xlabel("toroidal angle [deg]")
axes[0].set_ylabel("information gain / max")
axes[0].legend()
figure.tight_layout()
figure.savefig(PLOTS / "if_placement_gain_vs_toroidal_angle.pdf")

In [ ]:
figure, axis = plt.subplots(figsize=(4.5, 3))
for n_chords, rows in best.groupby("n_chords"):
    axis.plot(
        rows.toroidal_angle_deg,
        rows.direction_angle_deg,
        "o-",
        label=f"{n_chords} chord(s)",
    )
axis.set_xlabel("toroidal angle [deg]")
axis.set_ylabel("best chord direction [deg]")
axis.legend()
figure.tight_layout()
figure.savefig(PLOTS / "if_placement_best_direction.pdf")

## 2. Best and worst plane: optimized layout against the initial guess

The initial guess is the unoptimized layout in the same plane: vertical chords spread evenly across the plasma (one through the middle, or two at a third and two thirds of its width), scored with the same objective (`run_study.initial_guess`, about a minute: it loads the surrogate).

In [ ]:
cases = []
for n_chords, rows in best.groupby("n_chords"):
    extremes = {
        "best": rows.loc[rows.information_gain_bits.idxmax()],
        "worst": rows.loc[rows.information_gain_bits.idxmin()],
    }
    guesses = run_study.initial_guess(
        PATHS,
        tuple(float(row.toroidal_angle_deg) for row in extremes.values()),
        int(n_chords),
    )
    for (label, optimized), (_, guess) in zip(extremes.items(), guesses.iterrows()):
        cases.append((f"{label} plane, {n_chords} chord(s)", optimized, guess))
pd.DataFrame(
    [
        {
            "case": case,
            "toroidal_angle_deg": optimized.toroidal_angle_deg,
            "optimized_bits": optimized.information_gain_bits,
            "initial_guess_bits": guess.information_gain_bits,
        }
        for case, optimized, guess in cases
    ]
)

In [ ]:
reference = sweep_utils.reference_truth(
    PATHS.data_directory / f"{PATHS.dataset}.parquet",
    PATHS.data_directory / f"{PATHS.dataset}_settings.json",
)

# One row per chord count, drawn as the chord layouts of the calibration-bound study:
# best plane, then worst, each initial guess before its optimized layout.
for n_chords in sorted({int(optimized.n_chords) for _, optimized, _ in cases}):
    panels = [
        (f"{case.split(',')[0]}:\n{stage}", row)
        for case, optimized, guess in cases
        if int(optimized.n_chords) == n_chords
        for stage, row in (("initial guess", guess), ("optimized", optimized))
    ]
    figure, axes_row = pubfig.subplots(1, len(panels), width="double", aspect=2.0)
    for axes, (title, row) in zip(axes_row, panels):
        design = run_study.design_point(row)
        _, surface = geometry_utils.cross_section_and_surface(
            reference.coefficients,
            np.radians(design.toroidal_angle_deg),
            design.surface_offset,
        )
        plots_diagnostics_forward_models.draw_interferometer_layout(
            axes,
            reference.coefficients,
            geometry_utils.chords_from_surface_angles(
                surface, np.asarray(design.chord_surface_angles)
            ),
            design,
        )
        axes.set_title(
            f"{title}\n{row.toroidal_angle_deg:.1f} deg, "
            f"{row.information_gain_bits:.2f} bit",
            fontsize="small",
        )
    for axes in axes_row[1:]:
        axes.set_ylabel("")
    figure.legend(
        handles=axes_row[0].get_legend_handles_labels()[0],
        ncol=3,
        loc="outside lower center",
    )
    pubfig.save(
        figure,
        str(PLOTS / f"if_placement_optimized_vs_initial_guess_{n_chords}_chords"),
    )

## 3. Gain per truth at the best layouts

With the mean and the standard error of the mean over the truths, written to `tables/`.

In [ ]:
overall = best.loc[best.groupby("n_chords").information_gain_bits.idxmax()]
per_truth = overall.set_index("n_chords")[
    [column for column in overall.columns if column.startswith("bits_")]
].T
per_truth.index = [name.removeprefix("bits_") for name in per_truth.index]
gain_table = pd.concat(
    [
        per_truth,
        pd.DataFrame({"mean": per_truth.mean(), "standard error": per_truth.sem()}).T,
    ]
)
gain_table.round(2).to_csv(SCAN.parent / "if_placement_gain_per_truth.csv")
gain_table.round(2)

## 4. Inference at the best optimized layout and at the worst initial guess

`map_laplace` on every truth with the interferometer alone, no scattering volume, at the optimized layout of the best plane and at the initial guess with the lowest gain of section 2, for one and two chords (`run_study.compared_layouts`): 32 runs, run by `python run_placement_inference.py --study if_placement_study --workers 4` in the folder above, which stores them under this study's name and writes `inference_ids.json` beside this notebook.

The table compares the worst layout (initial guess) with the best (optimized) of each chord count: the expected information gain of the layout, and the RMSE of the reported mean (percent of the truth's peak) and the median half-width of the 5 to 95 percent band (percent of the local value) of `ne` and `Te`, mean and standard error over the eight truths (`run_placement_inference.accuracy_table`). The per-run values, with whether each MAP converged, go to `tables/` beside the summary.


In [ ]:
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.results.diagnostics_placement_optimization import (  # noqa: E501
    run_placement_inference,
)

LAYOUTS = {
    "1_chords_initial_guess": "1 chord, initial guess",
    "1_chords_optimized": "1 chord, optimized",
    "2_chords_initial_guess": "2 chords, initial guess",
    "2_chords_optimized": "2 chords, optimized",
}
run_ids = sweep_utils.current_ids(
    run_placement_inference.ids_file(SCAN.parents[1].name)
)
assert run_ids, (
    "no stored runs yet: run `python run_placement_inference.py "
    "--study if_placement_study --workers 4` in the folder above first"
)
per_run = run_placement_inference.accuracy_table(run_ids, list(LAYOUTS))
per_run.round(3).to_csv(SCAN.parent / "if_placement_inference_per_run.csv", index=False)
converged = per_run.drop_duplicates(["layout", "truth_name"]).map_converged
print(f"{int(converged.sum())} of {len(converged)} MAP runs converged")
compared = run_study.compared_layouts(table, PATHS)
summary = run_placement_inference.accuracy_summary(per_run)
summary.insert(
    0,
    ("expected gain", "bit"),
    [round(float(compared[layout].information_gain_bits), 2) for layout in LAYOUTS],
)
summary = summary.rename(index=LAYOUTS)
summary.to_csv(SCAN.parent / "if_placement_inference_summary.csv")
summary

## 5. Gain, error and band against the number of chords

The exhaustive scan of section 1 repeated for one to six parallel chords, written by `python run_study.py --chord-counts 1 2 3 4 5 6 --output tables/if_placement_scan_chord_counts.csv` in this folder (about three hours, 3 GB). `map_laplace` then runs on every truth at the optimized layout of three to six chords, in the folder above,

```
python run_placement_inference.py --study if_placement_study \
    --scan if_placement_study/tables/if_placement_scan_chord_counts.csv \
    --layouts 3_chords_optimized 4_chords_optimized 5_chords_optimized \
    6_chords_optimized --workers 4
```

into the manifest of section 4, whose one- and two-chord runs are the same layouts. The figure gives, per chord count, the expected gain of the optimized layout and the `ne` RMSE and band half-width of its inference, mean and standard error over the eight truths as in section 4; the table goes to `tables/`.

In [ ]:
assert run_study.CHORD_COUNT_OUTPUT.exists(), (
    "no chord-count scan yet: run `python run_study.py --chord-counts 1 2 3 4 5 6 "
    "--output tables/if_placement_scan_chord_counts.csv` in this folder first"
)
chord_count_scan = pd.read_csv(run_study.CHORD_COUNT_OUTPUT)
per_plane = chord_count_scan.loc[
    chord_count_scan.groupby(
        ["toroidal_angle_deg", "n_chords"]
    ).information_gain_bits.idxmax()
]
optimized = per_plane.loc[
    per_plane.groupby("n_chords").information_gain_bits.idxmax()
].set_index("n_chords")
# Sections 1 to 4 read the one- and two-chord optima off the first scan.
for n_chords, row in overall.set_index("n_chords").iterrows():
    assert np.isclose(
        optimized.loc[n_chords].information_gain_bits, row.information_gain_bits
    )
optimized[
    ["toroidal_angle_deg", "direction_angle_deg", "information_gain_bits"]
].assign(
    added_bits=optimized.information_gain_bits.diff().fillna(
        optimized.information_gain_bits
    )
).round(2)

In [ ]:
CHORD_COUNT_LAYOUTS = [f"{n_chords}_chords_optimized" for n_chords in optimized.index]
missing = [
    layout
    for layout in CHORD_COUNT_LAYOUTS
    if not any(label.endswith(f"_{layout}_seed00") for label in run_ids)
]
assert not missing, (
    f"no runs of {missing} yet: run the `run_placement_inference.py` command above "
    "in the folder above first"
)
per_run_chord_count = run_placement_inference.accuracy_table(
    run_ids, CHORD_COUNT_LAYOUTS
)
density = per_run_chord_count[per_run_chord_count.profile == "electron_density"]
statistics = density.groupby("layout", sort=False)[
    ["information_gain_bits", *run_placement_inference.METRICS]
].agg(["mean", "sem"])
chord_count_summary = pd.DataFrame(
    {
        "toroidal_angle_deg": optimized.toroidal_angle_deg,
        "direction_angle_deg": optimized.direction_angle_deg,
        "expected_gain_bits": optimized.information_gain_bits,
        "map_gain_bits": statistics["information_gain_bits", "mean"].to_numpy(),
        "ne_error_percent": statistics["error_percent", "mean"].to_numpy(),
        "ne_error_standard_error": statistics["error_percent", "sem"].to_numpy(),
        "ne_uncertainty_percent": statistics["uncertainty_percent", "mean"].to_numpy(),
        "ne_uncertainty_standard_error": statistics[
            "uncertainty_percent", "sem"
        ].to_numpy(),
    },
    index=optimized.index,
)
chord_count_summary.round(2).to_csv(
    run_study.CHORD_COUNT_OUTPUT.parent / "if_placement_chord_count_summary.csv"
)

color = pubfig.PALETTE[0]
figure, (gain_axes, error_axes, band_axes) = pubfig.subplots(
    1, 3, width="double", aspect=0.8
)
gain_axes.plot(
    chord_count_summary.index,
    chord_count_summary.expected_gain_bits,
    "o-",
    color=color,
)
gain_axes.set_ylabel(r"$\overline{\mathrm{EIG}}$ [bits]")
for axes, column, label in (
    (error_axes, "ne_error", r"$n_e$ RMSE [%]"),
    (band_axes, "ne_uncertainty", r"$n_e$ band [%]"),
):
    axes.errorbar(
        chord_count_summary.index,
        chord_count_summary[f"{column}_percent"],
        yerr=chord_count_summary[f"{column}_standard_error"],
        fmt="o-",
        color=color,
        capsize=2,
    )
    axes.set_ylabel(label)
for axes in (gain_axes, error_axes, band_axes):
    axes.set_xlabel(r"$N_{\mathrm{LoS}}$")
    axes.set_xticks(chord_count_summary.index)
pubfig.save(figure, str(PLOTS / "if_placement_chord_count"))
chord_count_summary.round(2)